# Q1. Compute the posterior probability for Tomorrow

Tomorrow: Outlook=Rainy, Temperature=Mild, Humidity=High, Windy=True.

Priors:
- P(Yes)=9/14
- P(No)=5/14

Likelihoods:
- P(Rainy|Yes)=2/9, P(Mild|Yes)=4/9, P(High|Yes)=3/9, P(True|Yes)=3/9
- P(Rainy|No)=3/5, P(Mild|No)=2/5, P(High|No)=4/5, P(True|No)=3/5

Posterior:
P(Yes|x) = [P(Yes)×∏P(feature|Yes)] / ([P(Yes)×∏P(feature|Yes)] + [P(No)×∏P(feature|No)]) = 0.14637

Decision: No (Don’t Play)


In [3]:
import pandas as pd
import numpy as np

# Load dataset
pg = pd.read_csv("PlayGolf.csv")
pg["Windy"] = pg["Windy"].astype(bool)

# Observation for Tomorrow
x = {"Outlook":"Rainy", "Temperature":"Mild", "Humidity":"High", "Windy":True}

# Priors
counts = pg["PlayGolf"].value_counts()
prior_yes = counts["Yes"] / len(pg)
prior_no  = counts["No"]  / len(pg)

# Conditional probabilities
def cond_prob(df, target, feat, val):
    sub = df[df["PlayGolf"] == target]
    return (sub[feat] == val).mean()

terms_yes = {k: cond_prob(pg, "Yes", k, v) for k, v in x.items()}
terms_no  = {k: cond_prob(pg, "No",  k, v) for k, v in x.items()}

# Posterior
num_yes = prior_yes * np.prod(list(terms_yes.values()))
num_no  = prior_no  * np.prod(list(terms_no.values()))
post_yes = num_yes / (num_yes + num_no)

print("Posterior P(Play=Yes|x) =", round(post_yes, 5))
print("Decision:", "No (Don't Play)" if post_yes < 0.5 else "Yes (Play)")


Posterior P(Play=Yes|x) = 0.14637
Decision: No (Don't Play)


# Q2. Independence of Outlook and Humidity

Check whether Outlook and Humidity are independent.

Conditional probabilities:
- P(High|Sunny)=0.4
- P(High|Overcast)=0.5
- P(High|Rainy)=0.6
- Marginal P(High)=0.5

Because P(High|Outlook) ≠ P(High) for some Outlook values, Outlook and Humidity are **not independent**.


In [4]:
import pandas as pd

pg = pd.read_csv("PlayGolf.csv")

# Contingency table for P(Humidity | Outlook)
ct = pd.crosstab(pg["Outlook"], pg["Humidity"], normalize="index")
marg = pg["Humidity"].value_counts(normalize=True)

print("P(Humidity | Outlook):")
print(ct)
print("\nP(Humidity):")
print(marg.sort_index())


P(Humidity | Outlook):
Humidity  High  Normal
Outlook               
Overcast   0.5     0.5
Rainy      0.6     0.4
Sunny      0.4     0.6

P(Humidity):
Humidity
High      0.5
Normal    0.5
Name: proportion, dtype: float64


# Q1. Recode ordinal variables

Recode the following ordinal variables as ordered integers:

- Outlook: Sunny=0, Overcast=1, Rainy=2  
- Temperature: Cool=0, Mild=1, Hot=2

Do not change other variables in this step.


In [5]:
import pandas as pd

# Load dataset
pg = pd.read_csv("PlayGolf.csv")

# Recode ordinal variables
enc_outlook = {"Sunny":0, "Overcast":1, "Rainy":2}
enc_temp    = {"Cool":0, "Mild":1, "Hot":2}

pg_ord = pg.copy()
pg_ord["Outlook"] = pg_ord["Outlook"].map(enc_outlook)
pg_ord["Temperature"] = pg_ord["Temperature"].map(enc_temp)

# Display encoded sample
print(pg_ord[["Outlook","Temperature","Humidity","Windy","PlayGolf"]].head())


   Outlook  Temperature Humidity  Windy PlayGolf
0        2            2     High  False       No
1        2            2     High   True       No
2        1            2     High  False      Yes
3        0            1     High  False      Yes
4        0            0   Normal  False      Yes


# Q2. Convert remaining variables and train Naive Bayes

Convert the remaining categorical variables to binary:

- Humidity: Normal=0, High=1  
- Windy: False=0, True=1  

Use the encoded data to train a Categorical Naive Bayes model on PlayGolf.csv.


In [6]:
import pandas as pd
from sklearn.naive_bayes import CategoricalNB

# Load dataset
pg = pd.read_csv("PlayGolf.csv")

# Encode all variables
enc_outlook = {"Sunny":0, "Overcast":1, "Rainy":2}
enc_temp    = {"Cool":0, "Mild":1, "Hot":2}
enc_hum     = {"Normal":0, "High":1}

pg_enc = pd.DataFrame({
    "Outlook": pg["Outlook"].map(enc_outlook),
    "Temperature": pg["Temperature"].map(enc_temp),
    "Humidity": pg["Humidity"].map(enc_hum),
    "Windy": pg["Windy"].astype(str).map({"False":0,"True":1}),
    "PlayGolf": pg["PlayGolf"]
})

# Split features and label
X = pg_enc.drop("PlayGolf", axis=1)
y = (pg_enc["PlayGolf"] == "Yes").astype(int)

# Train Categorical Naive Bayes model
model = CategoricalNB(alpha=1e-10)
model.fit(X, y)

print("Model trained successfully.")


Model trained successfully.


# Q3. Predict PlayGolfNext.csv

Use the trained Categorical Naive Bayes model to predict for the next three days from PlayGolfNext.csv.

Show the predicted probability of Play=Yes and the final decision for each day.


In [7]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import CategoricalNB

# Load both datasets
pg = pd.read_csv("PlayGolf.csv")
pg_next = pd.read_csv("PlayGolfNext.csv")

# Encoding
enc_outlook = {"Sunny":0, "Overcast":1, "Rainy":2}
enc_temp    = {"Cool":0, "Mild":1, "Hot":2}
enc_hum     = {"Normal":0, "High":1}

def encode_df(df):
    return pd.DataFrame({
        "Outlook": df["Outlook"].map(enc_outlook),
        "Temperature": df["Temperature"].map(enc_temp),
        "Humidity": df["Humidity"].map(enc_hum),
        "Windy": df["Windy"].astype(str).map({"False":0,"True":1})
    })

# Train model
X = encode_df(pg)
y = (pg["PlayGolf"] == "Yes").astype(int)
model = CategoricalNB(alpha=1e-10).fit(X, y)

# Predict next days
X_next = encode_df(pg_next)
proba_yes = model.predict_proba(X_next)[:, 1]
pred = np.where(proba_yes >= 0.5, "Yes (Play)", "No (Don't Play)")

# Combine results
res = pg_next.copy()
res["P(Play=Yes)"] = proba_yes
res["Decision"] = pred

print(res)


                  Day   Outlook Temperature Humidity  Windy  P(Play=Yes)  \
0  Day After Tomorrow  Overcast        Cool     High  False     1.000000   
1            Tomorrow     Rainy        Mild     High   True     0.146370   
2               Today     Sunny         Hot   Normal  False     0.822368   

          Decision  
0       Yes (Play)  
1  No (Don't Play)  
2       Yes (Play)  


# Q4. Compare model prediction with manual calculation

Compare the model’s prediction for “Tomorrow”  
(Outlook=Rainy, Temperature=Mild, Humidity=High, Windy=True)  
with the manual result from Section 1.

Both results show P(Play=Yes) ≈ 0.146 → Decision: **No (Don’t Play)**.

Therefore, the manual calculation and the model prediction are consistent.


In [8]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import CategoricalNB

# Load dataset
pg = pd.read_csv("PlayGolf.csv")

# Encoding
enc_outlook = {"Sunny":0, "Overcast":1, "Rainy":2}
enc_temp    = {"Cool":0, "Mild":1, "Hot":2}
enc_hum     = {"Normal":0, "High":1}

def encode_df(df):
    return pd.DataFrame({
        "Outlook": df["Outlook"].map(enc_outlook),
        "Temperature": df["Temperature"].map(enc_temp),
        "Humidity": df["Humidity"].map(enc_hum),
        "Windy": df["Windy"].astype(str).map({"False":0,"True":1})
    })

# Train model
X = encode_df(pg)
y = (pg["PlayGolf"] == "Yes").astype(int)
model = CategoricalNB(alpha=1e-10).fit(X, y)

# Predict for "Tomorrow"
tomorrow = pd.DataFrame([{
    "Outlook": "Rainy", "Temperature": "Mild", "Humidity": "High", "Windy": True
}])
proba_tomorrow = model.predict_proba(encode_df(tomorrow))[:,1][0]

print("P(Play=Yes | Tomorrow) =", round(proba_tomorrow, 5))
print("Decision:", "No (Don't Play)" if proba_tomorrow < 0.5 else "Yes (Play)")


P(Play=Yes | Tomorrow) = 0.14637
Decision: No (Don't Play)


# Q5. Compare predictions and confirm consistency

Compare the predictions for “Tomorrow” and “Today”  
with the manual calculation and class example.

Results:
- Tomorrow → P(Play=Yes)=0.146 → No (Don’t Play)
- Today → P(Play=Yes)=0.822 → Yes (Play)

The model predictions are consistent with the manual calculation and the class example.


In [9]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import CategoricalNB

# Load data
pg = pd.read_csv("PlayGolf.csv")
pg_next = pd.read_csv("PlayGolfNext.csv")

# Encoding
enc_outlook = {"Sunny":0, "Overcast":1, "Rainy":2}
enc_temp    = {"Cool":0, "Mild":1, "Hot":2}
enc_hum     = {"Normal":0, "High":1}

def encode_df(df):
    return pd.DataFrame({
        "Outlook": df["Outlook"].map(enc_outlook),
        "Temperature": df["Temperature"].map(enc_temp),
        "Humidity": df["Humidity"].map(enc_hum),
        "Windy": df["Windy"].astype(str).map({"False":0,"True":1})
    })

# Train model
X = encode_df(pg)
y = (pg["PlayGolf"] == "Yes").astype(int)
model = CategoricalNB(alpha=1e-10).fit(X, y)

# Predict all days in PlayGolfNext.csv
X_next = encode_df(pg_next)
proba_yes = model.predict_proba(X_next)[:,1]
pred = np.where(proba_yes >= 0.5, "Yes (Play)", "No (Don't Play)")

res = pg_next.copy()
res["P(Play=Yes)"] = proba_yes
res["Decision"] = pred

# Show only Today and Tomorrow
print(res.loc[res["Day"].isin(["Today","Tomorrow"])])


        Day Outlook Temperature Humidity  Windy  P(Play=Yes)         Decision
1  Tomorrow   Rainy        Mild     High   True     0.146370  No (Don't Play)
2     Today   Sunny         Hot   Normal  False     0.822368       Yes (Play)
